In [39]:
from typing import List, Any
import os
import weaviate
import json
import pandas as pd
from langchain_weaviate import WeaviateVectorStore
from weaviate.classes.query import Filter
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown
from dotenv import load_dotenv


In [40]:
load_dotenv('../.env.example/.env')

True

In [41]:
# weaviate Keys
WEAVIATE_URL = os.environ["WEAVIATE_URL"]
WEAVIATE_API_KEY = os.environ["WEAVIATE_API_KEY"]

In [42]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=WEAVIATE_API_KEY,
)

In [43]:
weaviate_client.close()

In [44]:
weaviate_client.connect()

In [45]:
weaviate_client.is_live()

True

# weaviate collection

In [ ]:
Euro_Laws = weaviate_client.collections.use("Euro_Laws")

In [ ]:
eur_docs_hybrid = weaviate_client.collections.get("Euro_Laws_hybrid")

In [46]:
# To get the full doc form Euro_Law_Documents collection
eur_docs = weaviate_client.collections.get("Euro_Law_Documents")

In [8]:
weaviate_client.collections.list_all().get('Euro_Law_Documents')

_CollectionConfigSimple(name='Euro_Law_Documents', description=None, generative_config=None, properties=[_Property(name='celex', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=None, vectorizer=None, vectorizer_configs={}), _Property(name='full_doc', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=None, vectorizer=None, vectorizer_configs={}), _Property(name='status', description=None, data_type=<DataType.TEXT: 'text'>, index_filterable=True, index_range_filters=False, index_searchable=True, nested_properties=None, tokenization=<Tokenization.WORD: 'word'>, vectorizer_config=None, vectorizer=None, vectorizer_configs={}), _Property(name='act_type', description=None, dat

# Embeddings Model class fix for langchain

In [37]:
class SentenceTransformersEmbeddings:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # returns a list of embeddings for documents
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        # returns a single embedding for a query
        return self.model.encode([text])[0].tolist()

In [38]:
embedding_model = SentenceTransformersEmbeddings('sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 502.47it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [39]:
quer_embed = embedding_model.embed_query("rape sentencing guidelines, punishment for rape, statutory penalties for rape, judicial discretion in rape cases")

# Use weaviate to search chunks with vector search only

In [ ]:
response = Euro_Laws.query.near_vector(
    near_vector= quer_embed,
    limit=5
)

In [11]:
for obj in response.objects:
    print (obj.properties)

{'act_name': 'Directive 2012/29/EU of the European Parliament and of the Council of 25Ã\x82 October 2012 establishing minimum standards on the rights, support and protection of victims of crime, and replacing Council Framework Decision 2001/220/JHA', 'total_chunks': 31, 'status': 'In Force', 'legal_basis': '12010E082; 12010E294', 'eurovoc': 'crime against individuals; restorative justice; aid for victims; access to justice; AFSJ', 'act_type': 'Directive', 'celex': '32012L0029', 'chunk_number': 9, 'document_length': 78554, 'authors': 'European Parliament; European Council', 'subject_matter': 'criminal law;  justice;  European construction', 'cites': 'dec_framw/2002/475; 52012XX0209%2802%29; 32001R45; dec_framw/2008/977; 42000A0712%2801%29; 52010XG0504%2801%29; dec_framw/2009/948; 52009IP0098%2801%29; 32011L99; 32011L93; 52011IP0127; 32011G0628%2801%29; 32011L36', 'text': "competent authorities are aware of the victim and throughout criminal proceedings and for an appropriate time after 

In [17]:
celex_ids = []

for obj in response.objects:
    celex_ids.append(obj.properties['celex'])

celex_ids

['32011L0093', '32012L0029', '32005F0214', '32019D0417', '32012L0029']

In [13]:
celex_ids = []

for obj in response.objects:
    celex_ids.append(obj.properties['celex'])

celex_ids

['32012L0029', '32005F0214', '32011L0093', '32019D0417', '32008F0947']

# Weaviate with LangChain

In [10]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws_hybrid",
    text_key="text",
    embedding = embedding_model
)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")

In [21]:
docs[0].metadata.keys()

dict_keys(['total_chunks', 'document_length', 'eurovoc', 'chunk_number', 'subject_matter', 'celex', 'additional_info', 'authors', 'act_type', 'legal_basis', 'status', 'treaty', 'cites', 'act_name'])

# to get the full doc by celex

In [9]:
def get_full_doc_weaviate(celex_id):

    response = eur_docs.query.fetch_objects(
    filters=Filter.by_property("celex").equal(celex_id),
    limit=1
)
    r = response.objects[0].properties

    # r is a dict with key, values for each doc
    return r    

In [10]:
def search_docs(query):
    celex_ids = []
    full_doc_info = """ """

    weaviate_client.connect()

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):
        f_doc_meta = get_full_doc_weaviate(celex_id)

        full_doc_info += f"""

        doc {i} :

        'celex': {f_doc_meta['celex']}
        'status': {f_doc_meta['status']}
        'act_type': {f_doc_meta['act_type']}
        'treaty': {f_doc_meta['treaty']}

        full_doc :

        {f_doc_meta['full_doc']}
{"=="*15} "END OF DOC" {"=="*15}
        """
    weaviate_client.close()
    
    return full_doc_info    

In [9]:
def get_embedding(query):
    embedding_model = SentenceTransformersEmbeddings("sentence-transformers/all-mpnet-base-v2")
    embed_query = embedding_model.embed_query(query)

    return embed_query

# hybrid search

In [10]:
def search_docs_hybrid(query):
    full_chunks_info = """ """

    # for debugging purpose
    weaviate_client.connect()

    # embed query
    embed_query = get_embedding(query)

    # Search docs using vectorstore
    response = eur_docs_hybrid.query.hybrid(
        query= query,
        vector= embed_query,
        alpha=0.5,
        limit=5
    )

    for i , obj in enumerate(response.objects):

        full_chunks_info += f"""

        chunk {i} :

        'celex': {obj.properties['celex']}
        'status': {obj.properties['status']}
        'act_type': {obj.properties['act_type']}
        'treaty': {obj.properties['treaty']}

        full_chunk :

        {obj.properties['text']}
{"=="*15} "END OF DOC" {"=="*15}
        """
        # for debugging purpose
        if len(full_chunks_info) > 5:
            print("docs found and retrieved successfully")
    print(len(full_chunks_info))

    weaviate_client.close()      

    return full_chunks_info  

In [ ]:
display(Markdown(search_docs_hybrid("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")))

In [ ]:
response = eur_docs_hybrid.query.hybrid(
    query="rape sentencing guidelines, punishment for rape, statutory penalties for rape, judicial discretion in rape cases",
    vector= quer_embed,
    alpha=0.5,
    limit=5
)

In [19]:
response.objects[0].properties

{'authors': 'European Commission',
 'treaty': 'TEC (1992)',
 'text': "Avis juridique important|31995D023295/232/EC: Commission Decision of 27 June 1995 on the organization of a temporary experiment under Council Directive 69/208/EEC in order to establish conditions to be satisfied by the seed of hybrids and varietal associations of swede rape and turnip rape Official Journal L 154 , 05/07/1995 P. 0022 - 0025COMMISSION DECISION of 27 June 1995 on the organization of a temporary experiment under Council Directive 69/208/EEC in order to establish conditions to be satisfied by the seed of hybrids and varietal associations of swede rape and turnip rape (95/232/EC)THE COMMISSION OF THE EUROPEAN COMMUNITIES, Having regard to the Treaty establishing the European Community, Having regard to Council Directive 69/208/EEC of 30 June 1969 on the marketing of seed of oil and fibre plants (1), as last amended by the Act of Accession of Austria, Finland and Sweden, and in particular Article 12a thereo

In [14]:
for obj in response.objects:
    print (obj.properties)

{'authors': 'European Commission', 'treaty': 'TEC (1992)', 'text': "Avis juridique important|31995D023295/232/EC: Commission Decision of 27 June 1995 on the organization of a temporary experiment under Council Directive 69/208/EEC in order to establish conditions to be satisfied by the seed of hybrids and varietal associations of swede rape and turnip rape Official Journal L 154 , 05/07/1995 P. 0022 - 0025COMMISSION DECISION of 27 June 1995 on the organization of a temporary experiment under Council Directive 69/208/EEC in order to establish conditions to be satisfied by the seed of hybrids and varietal associations of swede rape and turnip rape (95/232/EC)THE COMMISSION OF THE EUROPEAN COMMUNITIES, Having regard to the Treaty establishing the European Community, Having regard to Council Directive 69/208/EEC of 30 June 1969 on the marketing of seed of oil and fibre plants (1), as last amended by the Act of Accession of Austria, Finland and Sweden, and in particular Article 12a thereof,

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="hybrid",
    search_kwargs={
        "k": 5,
        "alpha": 0.5
    }
)
docs = retriever.invoke("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")

# Get relevent docs from reranked celex

In [ ]:
def get_full_docs_celex(celex_ids):
    full_doc_info = """ """

    # open weaviate client
    weaviate_client.connect()

    # Remove duplicates from celex_ids list
    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):
        f_doc_meta = get_full_doc_weaviate(celex_id)

        full_doc_info += f"""

        doc {i} :

        'celex': {f_doc_meta['celex']}
        'status': {f_doc_meta['status']}
        'act_type': {f_doc_meta['act_type']}
        'treaty': {f_doc_meta['treaty']}

        full_doc :

        {f_doc_meta['full_doc']}
{"=="*15} "END OF DOC" {"=="*15}
        """
        # for debugging purpose
        if len(full_doc_info) > 5:
            print("docs found and retrieved successfully")
    print(len(full_doc_info))

    #open weaviate client
    weaviate_client.close()      

    return full_doc_info

In [ ]:
celex_list = ['32018R1725', '32018D1962']

In [ ]:
display(Markdown(get_full_docs_celex(celex_list)))

# End


In [26]:
weaviate_client.close()

In [ ]:
display(Markdown(search_docs("trespassing sentencing penalties fines imprisonment statutory sanctions jurisdiction")))

In [ ]:
display(Markdown(search_docs("Drug dealing Sentences")))

In [7]:
laws = pd.read_csv('dataset/act_raw_text_with_4meta.csv')

In [17]:
laws.columns


Index(['Unnamed: 0', 'CELEX', 'Status', 'Act_type', 'Treaty', 'act_raw_text'], dtype='str')

In [8]:
def get__embeddings(enhanced_query):
    query_embedding = embedding_model.embed_query(enhanced_query)

    return query_embedding

In [9]:
query_embedding = get__embeddings("driving without license penalty")

In [45]:
weaviate_client.is_live()

The `WeaviateClient` is closed. Run `client.connect()` to (re)connect!


False

In [ ]:
eu = weaviate_client.collections.use("Euro_Laws")
response = eu.query.near_vector(
    near_vector= query_embedding, 
    limit=5
)

for obj in response.objects:
    print(json.dumps(obj.properties, indent=10))  # Inspect the results

In [11]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("driving without license penalty")
docs

```
'celex': , 'status': 

'act_type' , 'treaty':

```

# To Get celex ids from the retrived chunks

In [15]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(i.metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids

get_celex_ids(docs)  

['32015L0413', '32006L0126', '32009R1072', '31980L1263']

In [22]:
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]

'In Force'

# Search with enchanced query to get a sting has the relvent chunks with its info

In [18]:
def search_docs(query):
    celex_ids = []
    full_doc_info = """ """

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list(set(celex_ids))    

    for i , celex_id in enumerate(celex_ids):

        full_doc_info += f"""

        doc {i} :

        'celex': {laws[laws['CELEX'] == celex_id]['CELEX'].iloc[0]}
        'status': {laws[laws['CELEX'] == celex_id]['Status'].iloc[0]}
        'act_type': {laws[laws['CELEX'] == celex_id]['Act_type'].iloc[0]}
        'treaty': {laws[laws['CELEX'] == celex_id]['Treaty'].iloc[0]}

        full_doc :

        {laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]}
{"=="*15} "END OF DOC" {"=="*15}
        """

    return full_doc_info    



In [ ]:
display(Markdown(search_docs("driving without license penalty"))) 

In [23]:
laws[laws['CELEX'] == '32015L0413']['CELEX'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Status'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Act_type'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['Treaty'].iloc[0]
laws[laws['CELEX'] == '32015L0413']['act_raw_text'].iloc[0]


"13.3.2015 EN Official Journal of the European Union L 68/9 DIRECTIVE (EU) 2015/413 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 11 March 2015 facilitating cross-border exchange of information on road-safety-related traffic offences (Text with EEA relevance) THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on the Functioning of the European Union, and in particular Article 91(1)(c) thereof, Having regard to the proposal from the European Commission, After transmission of the draft legislative act to the national parliaments, Having regard to the opinion of the European Economic and Social Committee (1), After consulting the Committee of the Regions, Acting in accordance with the ordinary legislative procedure (2), Whereas: (1) Improving road safety is a prime objective of the Union's transport policy. The Union is pursuing a policy to improve road safety with the objective of reducing fatalities, injuries and material damage. An important e

In [ ]:
search_docs("Drug dealing Sentences")

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("Drug dealing Sentences")

retrived = []

for i , doc in enumerate (docs):
    retrived.append([f"Doc{i}:"  doc.metadata['status','treaty','act_type','celex']])


[Document(metadata={'subject_matter': 'sources and branches of the law;  European Union law;  justice;  criminal law', 'status': 'In Force', 'legal_basis': '12002M031; 12002M034', 'additional_info': 'CNS 2001/0114', 'chunk_number': 1, 'eurovoc': 'penal code; Community law - national law; criminal procedure; penalty; drug traffic', 'treaty': 'TEU (1992)', 'act_type': 'Decision_FRAMW', 'cites': 'joint_action/1997/396; 31999Y0123%2801%29; joint_action/1998/733', 'celex': '32004F0757', 'authors': 'European Council', 'act_name': 'Council Framework Decision 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking', 'document_length': 13663, 'total_chunks': 6}, page_content="11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the 

In [ ]:
retrived = []

for doc in docs:
    retrived.append({doc.metadata['status','treaty']})

In [45]:
docs[0].metadata['celex']

'32004F0757'

In [ ]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(docs[i].metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids    

# chat Template

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI."),
    ("human", "Answer this question: {question}")
])

formatted = prompt.invoke({"question": "What is AI?"})
print(formatted)

messages=[SystemMessage(content='You are a helpful AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Answer this question: What is AI?', additional_kwargs={}, response_metadata={})]


In [9]:
laws[laws['CELEX'] == '31997R2046']['act_raw_text'].iloc[0]   

"Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social de

# to get doc by text length

In [4]:
import sys
sys.path.append("..")

from src.core.utils import count_tokens
from src.retrieval.retrieval import get_full_docs_celex

Initializing vector store...
Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:06<00:00, 29.73it/s, Materializing param=pooler.dense.weight]                         
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Initializing vector store...


In [5]:
def get_full_doc_weaviate(text_length):
    weaviate_client.connect()

    response = eur_docs.query.fetch_objects(
    filters=Filter.by_property("text_length").greater_or_equal(text_length),
    limit=1
)
    r = response.objects[0].properties

    weaviate_client.close()
    # r is a dict with key, values for each doc
    return r    

In [48]:
weaviate_client.connect()

In [147]:
d = get_full_doc_weaviate(197850)

In [148]:
celex= str(d.get("celex"))
celex

'32003R1782'

In [65]:
doc = d.get("full_doc")

In [10]:
words = doc.split()
print(count_tokens(doc))
print(len(words))

535218
380908


In [11]:
docs=[doc]

In [137]:
def chunk_full_docs_for_summrize(celex_ids):

    chunks: list[list] = []

    full_docs, docs_list = get_full_docs_celex(celex_ids)

    if count_tokens(full_docs) > 20000:

        # chunk each doc into a list of chunks
        for doc in docs_list:

            if count_tokens(doc) < 20000:
                chunks.append([doc])

            else:

                doc_tokens_count = count_tokens(doc)
                words_in_doc = doc.split()
                no_words_doc = len(words_in_doc)

                split_coefficient: float = no_words_doc / doc_tokens_count
                split_size: int = int(19000 * split_coefficient)

                # For debugging purpuse
                print(f"doc_tokens_count: {doc_tokens_count}\nno_words_doc: {no_words_doc}\nsplit_coefficient: {split_coefficient}\nsplit_size: {split_size}")

                chunks.append([" ".join(words_in_doc[i:i+split_size]) for i in range(0, no_words_doc, split_size)])


    return chunks


In [149]:
chunks = chunk_full_docs_for_summrize(["31994R3384","32003R1782"])

docs found and retrieved successfully
docs found and retrieved successfully
278397
59648
doc_tokens_count: 42652
no_words_doc: 31362
split_coefficient: 0.7352996342492731
split_size: 13970


In [150]:
len(chunks)

2

In [151]:
for i, chunk in enumerate (chunks[0]):

    print(f"no of tokens in chunk {i}: {count_tokens(chunk)}")

no of tokens in chunk 0: 17830
no of tokens in chunk 1: 18263
no of tokens in chunk 2: 6549


In [152]:
chunks[0]

['doc 0 : \'celex\': 32003R1782 \'status\': Not in Force \'act_type\': Regulation \'treaty\': TEC (1992) full_doc : Avis juridique important|32003R1782Council Regulation (EC) No 1782/2003 of 29 September 2003 establishing common rules for direct support schemes under the common agricultural policy and establishing certain support schemes for farmers and amending Regulations (EEC) No 2019/93, (EC) No 1452/2001, (EC) No 1453/2001, (EC) No 1454/2001, (EC) 1868/94, (EC) No 1251/1999, (EC) No 1254/1999, (EC) No 1673/2000, (EEC) No 2358/71 and (EC) No 2529/2001 Official Journal L 270 , 21/10/2003 P. 0001 - 0069Council Regulation (EC) No 1782/2003of 29 September 2003establishing common rules for direct support schemes under the common agricultural policy and establishing certain support schemes for farmers and amending Regulations (EEC) No 2019/93, (EC) No 1452/2001, (EC) No 1453/2001, (EC) No 1454/2001, (EC) 1868/94, (EC) No 1251/1999, (EC) No 1254/1999, (EC) No 1673/2000, (EEC) No 2358/71 a